# 28. EfficientAD (model, trained on good tiles only)

EfficientAD (Batzner et al., WACV 2024). Three small networks: a frozen teacher that describes every patch, a student trained on good tiles to copy the teacher, and an autoencoder that rebuilds the teacher's features. On defects the student copies badly and the autoencoder disagrees with it; anomaly = both differences, each scaled with quantiles measured on good tiles, then averaged. No defect labels.

It **trains** in two stages (`teacher_steps`, then `steps`): expect many minutes per fabric on a CPU (the "All fabrics" cell trains one model per fabric); a GPU is much faster. It is built to be very fast at inference.

Differences from the paper (see `models/efficientad.py`): the teacher is distilled from the frozen `wide_resnet50_2` on the good tiles instead of from ImageNet, the optional ImageNet penalty is only used if you pass `penalty_images`, and the autoencoder is fully convolutional. Without the penalty the student can also copy the teacher on defects, which lowers detection: try passing other fabrics' tiles as `penalty_images`.

Model: `models/efficientad.py` (`METHOD_NAME = "efficientad"`). Evaluation: `evaluate.py`, same rules and same test tiles as every other experiment (README section 7). The model trains on part of the `good_cal` strips and the threshold comes from the other `good_cal` strips (`FIT_FRACTION`). Results are saved to `results/efficientad/`.

Needs `torch` and `torchvision`.

In [ ]:
import sys, importlib

sys.path.insert(0, "../src/methods")
import filters as F    # tile score, threshold and display helpers
import evaluate as E   # shared evaluation (README section 7)
import models as M     # learned models (needs torch + torchvision)
importlib.reload(F); importlib.reload(E); importlib.reload(M)  # pick up edits without restarting the kernel
print(M.device_info())

# ---- evaluation settings (same for every experiment) ----------------------
FABRIC = "02"           # fabric code = third part of the file name
METHOD_NAME = "efficientad"  # results go to results/<METHOD_NAME>/
THR_K = 4.0             # threshold = median + THR_K x 1.4826 x MAD of good_cal scores
MIN_AREA = 4            # damage mask: drop spots smaller than this (px); keep <= 4
SCORE_K = 25            # tile score = mean of the SCORE_K highest values of the smoothed map
SCORE_SIGMA = 2.0       # smoothing before scoring (px)
NORMALIZE = True        # score = strongest spot / the tile's own typical level (removes strip-to-strip bias)
FIT_FRACTION = 0.7      # this share of the good_cal strips trains the model, the rest sets the threshold
SETTINGS = dict(thr_k=THR_K, min_area=MIN_AREA, score_k=SCORE_K, score_sigma=SCORE_SIGMA,
                normalize=NORMALIZE, fit_fraction=FIT_FRACTION)

# ---- model settings (explained in models/efficientad.py) -----------------------
PARAMS = dict(
    size="small",                    # PDN size: "small" or "medium" (2x slower)
    out_channels=384,                # feature channels of the PDN (paper: 384)
    steps=3000,                      # student + autoencoder steps (paper: 70 000)
    teacher_steps=2000,              # steps to distill the teacher from the backbone
    batch_size=4,                    # tiles per step
    lr=1e-4,                         # Adam learning rate
    backbone="wide_resnet50_2",      # network the teacher learns from ("resnet18" is faster)
    penalty_images=None,             # optional list of grayscale images (>= 256 px) that are NOT this fabric
    smooth=4.0,                      # Gaussian sigma (px) of the final map
)
method = None
LAST = {}   # the last trained model is kept here (for the loss curve below)


def make_fit(**params):
    def fit(good_tiles):
        """Train on good tiles only. This is a real training loop: it takes minutes (see steps)."""
        fitted = M.fit_efficientad(good_tiles, **params)
        LAST["fitted"] = fitted
        return lambda tile: M.efficientad_map(tile, fitted)
    return fit


fit = make_fit(**PARAMS)

## Results for one fabric

In [ ]:
res = E.evaluate(method, FABRIC, fit=fit, **SETTINGS)
assert res is not None, f"not enough tiles for fabric {FABRIC}"
S = E.summarize(res)
E.print_summary(res, S)
print("saved to", E.save_results(res, METHOD_NAME))

### Training loss
The loss of the last model trained above. It should fall quickly and then flatten; if it is still falling at the end, raise the number of steps.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
loss = np.asarray(LAST["fitted"]["loss"])
k = max(1, len(loss) // 100)
plt.figure(figsize=(6, 3)); plt.plot(np.convolve(loss, np.ones(k) / k, mode="valid")); plt.xlabel("step"); plt.ylabel("loss (smoothed)")
plt.title(METHOD_NAME + ": training loss"); plt.tight_layout(); plt.show()

### Score distribution and ROC curve
Left: tile scores with the threshold. Right: recall vs false alarm rate for every possible threshold; the dot is the threshold chosen from good_cal. The curve uses test tiles and is for analysis only.

In [ ]:
E.plot_scores_roc(res, S)

### Recall per defect type

In [ ]:
E.print_by_type(res)

### Failure cases
Missed defects (lowest-scoring defect tiles) and false alarms (highest-scoring good_test / hard-negative tiles). Red = detected, green = ground-truth mask.

In [ ]:
E.show_failures(res, n=3)

### Example tiles
Largest defect, smallest defect (by mask area) and the last good tile of `FABRIC`.

In [ ]:
E.show_examples(res)

### One example per defect type
The dataset has 12 defect types (second part of the file name), spread over different fabrics. One tile per type: the clearest one (largest mask) from the fabric with the most tiles of that type. Each is judged by a model fitted and thresholded on its own fabric's good_cal strips (other fabrics are fitted here, so this cell takes longer). Red = detected, green = ground-truth mask.

In [ ]:
E.show_per_type(method, fit, res, **SETTINGS)

## All fabrics
Same settings on every fabric, one model and one threshold per fabric from its own good_cal strips. Columns: FAR on good_test and on hard negatives (hn), recall overall / small / large defects, strip-level FAR and recall, AUROC, IoU, ms per tile.

In [ ]:
summaries = E.all_fabrics(method, fit, METHOD_NAME, **SETTINGS)

## Hyperparameter tuning (optional)
Compares changes to `PARAMS` on `FABRIC` only (the development fabric). Copy the best row's values into `PARAMS` in the first code cell and rerun.

Picking a setting from this table uses the defect tiles of `FABRIC`, so the numbers of that fabric become optimistic. Judge the chosen setting on the **other** fabrics (the "All fabrics" table). Do not tune on every fabric separately.

In [ ]:
RUN_TUNING = False   # True = run the comparison below (each row trains the model again on FABRIC)
GRID = [             # each entry = a change to PARAMS
    {},
    {"steps": 1500, "teacher_steps": 1000},
    {"steps": 6000, "teacher_steps": 3000},
    {"size": "medium"},
    {"backbone": "resnet18"},
    {"out_channels": 256},
    {"lr": 5e-5},
    {"smooth": 2.0},
]
if RUN_TUNING:
    tuning = E.tune(make_fit, PARAMS, GRID, FABRIC, **SETTINGS)
